# LangChain Basics — Databricks Native

In [1]:
%run databricks_native_common.py

DATABRICKS_TOKEN, DATABRICKS_HOST, DATABRICKS_MODEL, (llm, llm_noreason), embeddings, vs_config = bootstrap_notebook()

c:\Users\akhawaja\git\cs4603\.venv-cs4603\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to: https://adb-2907158998162760.0.azuredatabricks.net/
Model endpoint: databricks-qwen35-122b-a10b
Vector Search index: cs4603.rag.docs_index


## 1. Direct LLM invocation

In [3]:
response = llm.invoke("What is Python in one sentence?")
print(response.content)

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Python (programming language).\n    *   Constraint: One sentence.\n    *   Goal: Define/Describe Python concisely.\n\n2.  **Identify Key Characteristics of Python:**\n    *   High-level.\n    *   Interpreted.\n    *   General-purpose.\n    *   Known for readability/simple syntax.\n    *   Versatile (web dev, data science, AI, scripting, etc.).\n    *   Dynamically typed.\n\n3.  **Draft Initial Ideas:**\n    *   Python is a programming language that is easy to read.\n    *   Python is a high-level, general-purpose programming language known for its clear syntax and versatility in fields like data science and web development.\n    *   Python is a popular interpreted programming language used by many developers because it is simple.\n    *   Python is a high-level, interpreted, general-purpose programming language renowned for its readable syntax and exten

## 2. Inspect the model name

In [4]:
print(llm.model)

databricks-qwen35-122b-a10b


## 3. Response structure

In [5]:
from pprintpp import pprint
pprint(response.response_metadata)

{
    'completion_tokens': 681,
    'finish_reason': 'stop',
    'model': 'qwen35-122b-a10b',
    'model_name': 'qwen35-122b-a10b',
    'prompt_tokens': 17,
    'total_tokens': 698,
    'usage': {
        'completion_tokens': 681,
        'prompt_tokens': 17,
        'total_tokens': 698,
    },
}


## 4. Reasoning vs No-Reasoning

`llm_noreason` uses `reasoning_effort="none"` — faster and cheaper for straightforward tasks.

In [6]:
r1 = llm.invoke("Explain recursion in one sentence.")
r2 = llm_noreason.invoke("Explain recursion in one sentence.")

print("With reasoning:   ", r1.content)
print("Without reasoning:", r2.content)

With reasoning:    [{"type": "reasoning", "summary": [{"type": "summary_text", "text": "Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Recursion (in computer science/mathematics).\n    *   Constraint: Explain it in *one sentence*.\n\n2.  **Identify Key Concepts of Recursion:**\n    *   A function/method calling itself.\n    *   Solving a problem.\n    *   Breaking it down into smaller similar sub-problems.\n    *   Base case (to stop infinite loops).\n\n3.  **Draft Initial Ideas:**\n    *   Recursion is when a function calls itself to solve a problem. (Too simple, misses the \"defining structure\" aspect)\n    *   Recursion is a method where the solution to a problem depends on solutions to smaller instances of the same problem. (Classic definition, accurate)\n    *   Recursion is a programming technique where a function calls itself in order to solve a problem by breaking it down into smaller, similar parts until a stopping condition is met. (A bit long, but accurat

## 5. Basic LCEL chain: prompt | llm | parser

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical explainer."),
    ("human", "Explain {topic} in exactly two sentences."),
])

chain = prompt | llm | StrOutputParser()

for topic in ["LangChain", "Prompt Engineering", "Vector Databases"]:
    print(f"\n--- {topic} ---")
    print(chain.invoke({"topic": topic}))


--- LangChain ---
[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: LangChain (a framework for developing applications powered by LLMs).\n    *   Constraint: Explain in *exactly* two sentences.\n    *   Role: Concise technical explainer.\n\n2.  **Identify Key Concepts of LangChain:**\n    *   Framework/Library.\n    *   Built for Large Language Models (LLMs).\n    *   Components: Chains, Agents, Retrieval, Memory, Prompts.\n    *   Purpose: Simplify development, orchestrate prompts, connect data sources, manage context.\n    *   Modularity/Composability.\n\n3.  **Drafting - Attempt 1:**\n    LangChain is a software framework designed to simplify the development of applications powered by large language models by providing modular components for prompts, memory, and data retrieval. It enables developers to orchestrate complex workflows that chain together multiple LLM calls and external APIs, making it